In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục chứa data giải nén trên Colab để đọc cho nhanh
!mkdir -p /content/dataset

# Giải nén data (lưu ý đường dẫn theo đúng tên thư mục của bạn)
!unzip -q "/content/drive/MyDrive/AIO_Homework/dence representation/data/data_train.zip" -d /content/dataset/
!unzip -q "/content/drive/MyDrive/AIO_Homework/dence representation/data/data_test.zip" -d /content/dataset/

Mounted at /content/drive


Sau khi chạy ô trên và cấp quyền, hãy xác định đường dẫn đến 2 tệp zip của bạn (thông thường nằm trong `/content/drive/MyDrive/...`). Thay đổi đường dẫn dưới đây cho đúng với tệp của bạn để giải nén:

In [2]:
import os
import re
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm import tqdm

In [3]:
import os
import re
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from tqdm import tqdm

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-záàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ_ \n]', ' ', text)
    tokens = text.split()
    return tokens

train_dir = "/content/dataset/" # Thư mục bạn giải nén lúc nãy
all_text = ""

# Lấy danh sách tất cả các file .txt trong thư mục để dùng với tqdm
file_paths = glob.glob(os.path.join(train_dir, '**', '*.txt'), recursive=True)

print("Đang đọc và tiền xử lý văn bản...")
# Đọc tất cả các file .txt trong thư mục với thanh tiến độ
for file_path in tqdm(file_paths, desc="Đọc file"): # Added tqdm here
    with open(file_path, 'r', encoding='utf-8') as f:
        all_text += f.read() + " "

tokens = preprocess_text(all_text)
vocab = set(tokens)
vocab_size = len(vocab)

word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}

print(f"Kích thước tập từ vựng (Vocabulary size): {vocab_size}")

Đang đọc và tiền xử lý văn bản...


Đọc file: 100%|██████████| 50000/50000 [25:02<00:00, 33.27it/s]


Kích thước tập từ vựng (Vocabulary size): 42297


In [4]:
window_size = 2
data = []

window_size = 2
data = []
for i in range(window_size, len(tokens) - window_size):
    context = [tokens[i - 2], tokens[i - 1], tokens[i + 1], tokens[i + 2]]
    target = tokens[i]
    context_idxs = [word2idx[w] for w in context]
    target_idx = word2idx[target]
    data.append((context_idxs, target_idx))

print(f"Tổng số mẫu huấn luyện: {len(data)}")

Tổng số mẫu huấn luyện: 4136681


In [12]:
class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWModel, self).__init__()
        # Lớp Embedding
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Lớp Linear
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, inputs):
        # inputs shape: (batch_size, context_size)
        embeds = self.embeddings(inputs)

        # Tính mean của các từ trong context: (batch_size, context_size, embedding_dim) -> (batch_size, embedding_dim)
        mean_embeds = torch.mean(embeds, dim=1)

        # Đưa qua lớp Linear
        out = self.linear(mean_embeds)

        # Nếu muốn output ra xác suất thực sự (để kiểm tra), bạn dùng F.log_softmax(out, dim=1)
        # Nhưng khi train với CrossEntropyLoss thì trả về 'out' nguyên bản là chuẩn nhất.
        return out

EMBEDDING_DIM = 100 # Đã tăng từ 50 lên 100
model = CBOWModel(vocab_size, EMBEDDING_DIM)

In [10]:
from sklearn.model_selection import train_test_split

class CBOWDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        context, target = self.data[idx]
        return torch.tensor(context, dtype=torch.long), torch.tensor(target, dtype=torch.long)

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
# Tỷ lệ 80% train, 20% validation
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)

print(f"Tổng số mẫu huấn luyện: {len(train_data)}")
print(f"Tổng số mẫu kiểm tra (validation): {len(val_data)}")

# Gói dữ liệu vào DataLoader, Batch Size = 256 (có thể tăng lên 512, 1024 tuỳ RAM)
BATCH_SIZE = 256
train_dataset = CBOWDataset(train_data)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = CBOWDataset(val_data)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Tổng số mẫu huấn luyện: 3309344
Tổng số mẫu kiểm tra (validation): 827337


In [13]:
import numpy as np

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pt', trace_func=print, min_delta=0):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement.
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
            path (str): Path for the checkpoint to be saved to.
                            Default: 'checkpoint.pt'
            trace_func (function): trace print function.
                            Default: print
            min_delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0. This parameter is renamed from `delta` to avoid confusion
                            and better reflect its purpose as minimum *improvement* required.
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf # Changed np.Inf to np.inf
        self.delta = min_delta # Use min_delta for the actual threshold
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            # self.save_checkpoint(val_loss, model) # No model saving in this example
        elif score < self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            # self.save_checkpoint(val_loss, model)
            self.counter = 0

# Chuyển mô hình sang GPU nếu có
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang train trên thiết bị: {device}")

model = CBOWModel(vocab_size, EMBEDDING_DIM).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

early_stopping = EarlyStopping(patience=3, min_delta=0.001)
epochs = 50

for epoch in range(epochs):
    # --- Training Phase ---
    model.train()
    total_train_loss = 0
    correct_train_preds = 0
    total_train_samples = 0

    progress_bar_train = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} (Train)', leave=False)

    for context_tensor, target_tensor in progress_bar_train:
        # Đẩy dữ liệu lên GPU (nếu có)
        context_tensor = context_tensor.to(device)
        target_tensor = target_tensor.to(device)

        model.zero_grad()
        log_probs = model(context_tensor)
        loss = loss_function(log_probs, target_tensor)

        loss.backward()
        optimizer.step()

        # Tính toán train metrics
        total_train_loss += loss.item() * context_tensor.size(0)
        preds = torch.argmax(log_probs, dim=1)
        correct_train_preds += (preds == target_tensor).sum().item()
        total_train_samples += context_tensor.size(0)

        # Hiển thị trên thanh tiến độ
        progress_bar_train.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{(correct_train_preds/total_train_samples):.4f}'
        })

    avg_train_loss = total_train_loss / total_train_samples
    avg_train_acc = correct_train_preds / total_train_samples

    # --- Validation Phase ---
    model.eval()
    total_val_loss = 0
    correct_val_preds = 0
    total_val_samples = 0

    with torch.no_grad(): # Không cần tính gradient trong quá trình validation
        progress_bar_val = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} (Val)', leave=False)
        for context_tensor, target_tensor in progress_bar_val:
            context_tensor = context_tensor.to(device)
            target_tensor = target_tensor.to(device)

            log_probs = model(context_tensor)
            loss = loss_function(log_probs, target_tensor)

            # Tính toán val metrics
            total_val_loss += loss.item() * context_tensor.size(0)
            preds = torch.argmax(log_probs, dim=1)
            correct_val_preds += (preds == target_tensor).sum().item()
            total_val_samples += context_tensor.size(0)

            # Hiển thị trên thanh tiến độ
            progress_bar_val.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{(correct_val_preds/total_val_samples):.4f}'
            })

    avg_val_loss = total_val_loss / total_val_samples
    avg_val_acc = correct_val_preds / total_val_samples

    print(f'Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc:.4f}')

    early_stopping(avg_val_loss) # Sử dụng val_loss cho early stopping
    if early_stopping.early_stop:
        print(f"Bắt đầu có dấu hiệu bão hòa. Dừng huấn luyện sớm tại epoch {epoch+1}!")
        break

Đang train trên thiết bị: cuda


Epoch 1/50 - Train Loss: 6.0283 | Train Acc: 0.0916 | Val Loss: 5.6421 | Val Acc: 0.1132


Epoch 2/50 - Train Loss: 5.4552 | Train Acc: 0.1210 | Val Loss: 5.4941 | Val Acc: 0.1239


Epoch 3/50 - Train Loss: 5.2967 | Train Acc: 0.1302 | Val Loss: 5.4284 | Val Acc: 0.1288


Epoch 4/50 - Train Loss: 5.2042 | Train Acc: 0.1357 | Val Loss: 5.3923 | Val Acc: 0.1315


...

Epoch 11/50 - Train Loss: 4.9264 | Train Acc: 0.1540 | Val Loss: 5.3259 | Val Acc: 0.1380


Epoch 12/50 - Train Loss: 4.9032 | Train Acc: 0.1555 | Val Loss: 5.3249 | Val Acc: 0.1383


Epoch 13/50 - Train Loss: 4.8823 | Train Acc: 0.1570 | Val Loss: 5.3245 | Val Acc: 0.1388
EarlyStopping counter: 1 out of 3


Epoch 14/50 - Train Loss: 4.8627 | Train Acc: 0.1585 | Val Loss: 5.3243 | Val Acc: 0.1387
EarlyStopping counter: 2 out of 3


Epoch 15/50 - Train Loss: 4.8449 | Train Acc: 0.1597 | Val Loss: 5.3254 | Val Acc: 0.1390
EarlyStopping counter: 3 out of 3
Bắt đầu có dấu hiệu bão hòa. Dừng huấn luyện sớm tại epoch 15!
